# Extraction TMDB — Notebook d'exploration

Ce notebook reprend, cellule par cellule, la logique du script `src/extract_api.py`.
L'objectif est de pouvoir **explorer** les données à chaque étape (voir le JSON brut, tester une fonction isolément, visualiser un DataFrame) avant de figer la logique dans le script final.

Pré-requis : un fichier `.env` à la racine du projet contenant `TMDB_API_KEY=votre_cle`.

## 1. Configuration

In [ ]:
import os
import json
import time

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("TMDB_API_KEY")
BASE_URL = "https://api.themoviedb.org/3"
LANGUE = "fr-FR"

assert API_KEY, "Clé API manquante : vérifiez votre fichier .env"

## 2. Films populaires

On commence par un seul appel, et on regarde la réponse brute avant d'écrire une fonction définitive.

In [ ]:
url = f"{BASE_URL}/movie/popular"
params = {"api_key": API_KEY, "language": LANGUE, "page": 1}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()
donnees_brutes = response.json()

donnees_brutes["results"][0]  # on regarde le premier film pour voir la structure

Maintenant qu'on a vu la structure, on formalise l'appel dans une fonction réutilisable.

In [ ]:
def get_popular_movies(page=1):
    """Récupère une page de films populaires. Retourne une liste de dicts (ou vide si erreur)."""
    url = f"{BASE_URL}/movie/popular"
    params = {"api_key": API_KEY, "language": LANGUE, "page": page}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as erreur:
        print(f"Erreur lors de la récupération des films populaires : {erreur}")
        return []

    return response.json().get("results", [])

In [ ]:
films_bruts = get_popular_movies(page=1)
print(f"{len(films_bruts)} films récupérés")

# Visualisation rapide sous forme de tableau (pratique pour repérer les champs intéressants)
pd.DataFrame(films_bruts).head()

## 3. Correspondance des genres

L'endpoint `popular` ne renvoie que des `genre_ids` (des nombres). On récupère la table de correspondance id -> nom.

In [ ]:
def get_genre_mapping():
    """Retourne un dict {id_genre: nom_genre}."""
    url = f"{BASE_URL}/genre/movie/list"
    params = {"api_key": API_KEY, "language": LANGUE}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as erreur:
        print(f"Erreur lors de la récupération des genres : {erreur}")
        return {}

    genres = response.json().get("genres", [])
    return {genre["id"]: genre["name"] for genre in genres}

In [ ]:
genres_mapping = get_genre_mapping()
genres_mapping

## 4. Détail d'un film

On teste sur un seul film avant de généraliser (ex. pour enrichir plus tard avec budget/durée).

In [ ]:
def get_movie_details(movie_id):
    """Retourne le détail complet d'un film (dict), ou None en cas d'erreur."""
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {"api_key": API_KEY, "language": LANGUE}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as erreur:
        print(f"Erreur lors de la récupération du film {movie_id} : {erreur}")
        return None

    return response.json()

In [ ]:
premier_id = films_bruts[0]["id"]
detail = get_movie_details(premier_id)
detail

## 5. Simplification et mise en forme

On ne garde que les champs utiles au projet, et on résout les `genre_ids` en noms de genres.

In [ ]:
def extraire_champs_utiles(film, genres_mapping):
    """Simplifie un film TMDB brut en un dict avec les champs utiles au projet."""
    genres_noms = [genres_mapping.get(gid, "Inconnu") for gid in film.get("genre_ids", [])]

    return {
        "id": film.get("id"),
        "titre": film.get("title"),
        "date_sortie": film.get("release_date"),
        "note_moyenne": film.get("vote_average"),
        "nombre_votes": film.get("vote_count"),
        "genres": genres_noms,
        "synopsis": film.get("overview"),
    }

In [ ]:
films_simplifies = [extraire_champs_utiles(film, genres_mapping) for film in films_bruts]

df_films = pd.DataFrame(films_simplifies)
df_films.head()

On peut maintenant explorer (tri par note, distribution des genres...) avant de figer quoi que ce soit dans le script.

In [ ]:
df_films.sort_values("note_moyenne", ascending=False).head(10)

## 6. Sauvegarde

Une fois satisfait du résultat, on sauvegarde en JSON — c'est ce même fichier que le script `extract_api.py` produit.

In [ ]:
def sauvegarder_json(donnees, chemin_fichier):
    with open(chemin_fichier, "w", encoding="utf-8") as fichier:
        json.dump(donnees, fichier, ensure_ascii=False, indent=2)
    print(f"{len(donnees)} film(s) sauvegardé(s) dans {chemin_fichier}")

sauvegarder_json(films_simplifies, "../data/raw/films_tmdb.json")

---
**Passage au script :** une fois cette exploration validée, la logique retenue est reportée telle quelle dans `src/extract_api.py`, sous forme de fonctions appelées depuis un bloc `if __name__ == "__main__":`. Le notebook reste utile pour explorer de nouvelles pistes (ex. `/movie/{id}/credits` pour le casting) avant de les intégrer au script.